# Network egress and SSRF lab

All experiments are offline: the notebook evaluates request-admission decisions, not real URLs.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from lab import FetchPolicy, allowed_url
policy = FetchPolicy(frozenset({'api.example.test'}))


## Vulnerable input cases

Scheme, host, redirect, and resolved address are separate checks.

In [ ]:
assert not allowed_url('http://api.example.test/policy', policy)['allow']
assert not allowed_url('https://localhost/admin', policy)['allow']
assert not allowed_url('https://api.example.test/policy', policy, resolved_ip='127.0.0.1')['allow']


## Retest with an authorized bounded request

A valid decision includes response limits for the separate fetch executor.

In [ ]:
decision = allowed_url('https://api.example.test/policy', policy, resolved_ip='8.8.8.8')
assert decision['allow'] and decision['max_bytes'] == 1_000_000
decision


## Production exercise

Add per-hop redirect validation, an egress-proxy policy receipt, and a test for cloud metadata addresses.